In [1]:
import time
import os
import cv2
import glob
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from PIL import Image, ImageDraw, ImageFont

In [2]:
def slice_geotiff(geotiff_path, output_size, slice_flag=True):
    slices = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width
        transform, crs, dtype, count = src.transform, src.crs, src.dtypes[0], src.count

        num_tiles_x = int(np.ceil(width / output_size))
        num_tiles_y = int(np.ceil(height / output_size))

        if not slice_flag:
            img_array = src.read()
            slices.append({
                "img_array": img_array,
                "transform": transform,
                "crs": crs,
                "dtype": dtype,
                "count": count,
                "position": (0, 0)
            })
            return slices

        for y in range(num_tiles_y):
            for x in range(num_tiles_x):
                x_start, y_start = x * output_size, y * output_size
                x_end, y_end = min(x_start + output_size, width), min(y_start + output_size, height)

                window = Window(x_start, y_start, x_end - x_start, y_end - y_start)
                slice_img = src.read(window=window)

                pad_x, pad_y = output_size - slice_img.shape[2], output_size - slice_img.shape[1]
                if pad_x or pad_y:
                    slice_img = np.pad(slice_img, ((0, 0), (0, pad_y), (0, pad_x)), mode='constant')

                new_transform = from_origin(
                    transform.c + x_start * transform.a,
                    transform.f + y_start * transform.e,
                    transform.a,
                    -transform.e
                )

                slices.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": crs,
                    "dtype": dtype,
                    "count": count,
                    "position": (y, x)
                })
    return slices


In [3]:
def overlay_segmentation_on_array(results, transform, img_shape, tif_image):
    object_data = []
    height, width, _ = img_shape

    for result in results:
        if result.masks is None:
            continue

        for mask, box, cls, conf in zip(result.masks.xy, result.boxes.xyxy, result.boxes.cls, result.boxes.conf):
            cls = int(cls.item())
            x1, y1, x2, y2 = map(int, box.tolist())

            mask_layer = np.zeros((height, width), dtype=np.uint8)
            if mask is not None and len(mask) > 0:
                cv2.fillPoly(mask_layer, [np.array(mask, dtype=np.int32)], 255)

            pixel_count = np.count_nonzero(mask_layer[y1:y2, x1:x2])
            area = pixel_count * (transform.a * abs(transform.e))

            lon, lat = transform * ((x1 + x2) / 2, (y1 + y2) / 2)

            object_data.append({
                "latitude": lat,
                "longitude": lon,
                "area": area,
                "pixel_count": pixel_count,
                "class": cls,
                "confidence": float(conf.item()),
                "tile_name": tif_image
            })

    return object_data

In [8]:
from tqdm.notebook import tqdm
import torch
import gc
from datetime import timedelta

def find_tif_images(folder_path):
    return glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)

from tqdm.notebook import tqdm
import torch
import gc
from datetime import timedelta

# Load model and setup device
model = YOLO("C:/Users/mza0288/Documents/CODE/runs/segment/train29/weights/best.pt")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
model.to(device)

# Parameters
output_size = 3000
batch_size = 8
folder_path = 'E:/Data/al_30cm_2021_transfer_2_of_2/'
# folder_path = 'C:/Users/mza0288/Documents/CODE/geotiff'

tif_images = find_tif_images(folder_path)
accumulate_object_infos = []

total_images = len(tif_images)

overall_bar = tqdm(
    enumerate(tif_images, 1), 
    total=total_images, 
    desc="Overall Progress",
    unit="image",
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
)

for idx, tif_image in overall_bar:
    image_start_time = time.time()
    slices = slice_geotiff(tif_image, output_size=output_size, slice_flag=True)
    slice_batches = [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]

    batch_bar = tqdm(
        slice_batches,
        desc=f"Processing {os.path.basename(tif_image)}",
        leave=False,
        unit="batch",
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
    )

    for batch in batch_bar:
        pil_imgs_batch = []
        for slice_data in batch:
            img_array = slice_data["img_array"]
            rgb_img = np.moveaxis(img_array[:3], 0, -1)
            pil_imgs_batch.append(Image.fromarray(rgb_img))

        results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)

        for slice_data, results in zip(batch, results_batch):
            object_info = overlay_segmentation_on_array(
                results,
                transform=slice_data["transform"],
                img_shape=rgb_img.shape,
                tif_image=os.path.basename(tif_image)
            )
            accumulate_object_infos.extend(object_info)

    # Save intermediate CSV after processing each image
    intermediate_csv = f"data/sub_2/intermediate_object_info_{idx}.csv"
    pd.DataFrame(accumulate_object_infos).to_csv(intermediate_csv, index=False)

    # Explicitly clear memory after each loop iteration
    del slices, slice_batches, batch_bar, pil_imgs_batch, results_batch, img_array, rgb_img
    torch.cuda.empty_cache()
    gc.collect()

    # Optional: print detailed timing per image processed
    image_time_taken = time.time() - image_start_time
    overall_bar.set_postfix({
        'Last_image_time': f"{timedelta(seconds=int(image_time_taken))}",
        'Images_left': total_images - idx,
    })

overall_bar.close()

final_csv_path = "data/final_2_2.csv"
pd.DataFrame(accumulate_object_infos).to_csv(final_csv_path, index=False)
print(f"\n✅ Final results saved to {final_csv_path}")


Using device: cuda


Overall Progress:   0%|          | 0/1493 [00:00<?, ?image/s]

Processing m_3308528_ne_16_030_20211108.tif:   0%|          | 0/9 [00:00<?, ?batch/s]

Processing m_3308501_nw_16_030_20211023.tif:   0%|          | 0/9 [00:00<?, ?batch/s]

Processing m_3308511_nw_16_030_20211027.tif:   0%|          | 0/9 [00:00<?, ?batch/s]

...

Processing m_3508863_sw_16_030_20211108.tif:   0%|          | 0/9 [00:00<?, ?batch/s]

Processing m_3508863_se_16_030_20211104.tif:   0%|          | 0/9 [00:00<?, ?batch/s]

Processing m_3508864_sw_16_030_20211104.tif:   0%|          | 0/9 [00:00<?, ?batch/s]

Processing m_3508864_se_16_030_20211104.tif:   0%|          | 0/9 [00:00<?, ?batch/s]


✅ Final results saved to data/final_2_2.csv


In [14]:


























from tqdm.notebook import tqdm
import torch
import gc
import glob
import os
import pandas as pd
import numpy as np
import time
from datetime import timedelta
from PIL import Image

# Parameters and paths
model_path = "C:/Users/mza0288/Documents/CODE/runs/segment/train29/weights/best.pt"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

output_size = 3000
batch_size = 8
folder_path = 'E:/Data/al_30cm_2021_transfer_1_of_2/'
intermediate_csv_file = "data/sub_1/intermediate_object_info_1404.csv"
final_csv_path = "data/final_1_2.csv"

# Ensure directories exist
os.makedirs(os.path.dirname(final_csv_path), exist_ok=True)

# Load model
model = YOLO(model_path).to(device)

# Helper functions
def find_tif_images(folder_path):
    return sorted(glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True))

# Load existing data
existing_df = pd.read_csv(intermediate_csv_file)
last_processed_file = existing_df['tile_name'].iloc[-1]
print(f"✅ Last processed file: {last_processed_file}")

# Determine files to process after last_processed_file
all_tif_images = find_tif_images(folder_path)
remaining_images = []
skip = True
for img in all_tif_images:
    if os.path.basename(img) == last_processed_file:
        skip = False  # Start processing from next file after this
        continue
    if not skip:
        remaining_images.append(img)

print(f"Total images: {len(all_tif_images)}, Remaining images to process: {len(remaining_images)}")

accumulate_object_infos = []

overall_bar = tqdm(
    enumerate(remaining_images, 1), 
    total=len(remaining_images), 
    desc="Overall Progress",
    unit="image",
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
)

for idx, tif_image in overall_bar:
    image_start_time = time.time()
    slices = slice_geotiff(tif_image, output_size=output_size, slice_flag=True)
    slice_batches = [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]

    batch_bar = tqdm(
        slice_batches,
        desc=f"{os.path.basename(tif_image)}",
        leave=False,
        unit="batch",
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
    )

    for batch in batch_bar:
        pil_imgs_batch = []
        for slice_data in batch:
            img_array = slice_data["img_array"]
            rgb_img = np.moveaxis(img_array[:3], 0, -1)
            pil_imgs_batch.append(Image.fromarray(rgb_img))

        results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)

        for slice_data, results in zip(batch, results_batch):
            object_info = overlay_segmentation_on_array(
                results,
                transform=slice_data["transform"],
                img_shape=rgb_img.shape,
                tif_image=os.path.basename(tif_image)
            )
            accumulate_object_infos.extend(object_info)

    # Append newly processed results to existing DataFrame after each image
    new_df = pd.DataFrame(accumulate_object_infos)
    existing_df = pd.concat([existing_df, new_df], ignore_index=True)

    # Incrementally update CSV
    existing_df.to_csv(intermediate_csv_file, index=False)

    # Clear accumulated objects after saving intermediate results
    accumulate_object_infos.clear()

    # Memory cleanup
    del slices, slice_batches, batch_bar, pil_imgs_batch, results_batch, img_array, rgb_img, new_df
    torch.cuda.empty_cache()
    gc.collect()

    # Show timing info
    image_time_taken = time.time() - image_start_time
    overall_bar.set_postfix({
        'Last_image_time': f"{timedelta(seconds=int(image_time_taken))}",
        'Images_left': len(remaining_images) - idx,
    })

overall_bar.close()

# Save final CSV file
existing_df.to_csv(final_csv_path, index=False)
print(f"\n✅ Final results saved to {final_csv_path}")


Using device: cuda
✅ Last processed file: m_3208612_nw_16_030_20211026.tif
Total images: 1987, Remaining images to process: 577


Overall Progress:   0%|          | 0/577 [00:00<?, ?image/s]

m_3208612_se_16_030_20211026.tif:   0%|          | 0/9 [00:00<?, ?batch/s]

m_3208612_sw_16_030_20211026.tif:   0%|          | 0/9 [00:00<?, ?batch/s]

m_3208613_ne_16_030_20211101.tif:   0%|          | 0/9 [00:00<?, ?batch/s]

...

m_3208864_ne_16_030_20211120.tif:   0%|          | 0/9 [00:00<?, ?batch/s]

m_3208864_nw_16_030_20211120.tif:   0%|          | 0/9 [00:00<?, ?batch/s]

m_3208864_se_16_030_20211120.tif:   0%|          | 0/9 [00:00<?, ?batch/s]

m_3208864_sw_16_030_20211120.tif:   0%|          | 0/9 [00:00<?, ?batch/s]


✅ Final results saved to data/final_1_2.csv
